# Loan Default Analysis & Predictive Modeling (SQL Server Pipeline)

In [1]:
import os
import sys
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array

os.environ["HADOOP_HOME"] = os.path.abspath("../hadoop")
os.environ["PATH"] += os.pathsep + os.path.join(os.environ["HADOOP_HOME"], "bin")
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

# Load SQL Server details from .env
load_dotenv(dotenv_path="../.env")
host = os.environ.get("SQLSERVER_HOST", "localhost")
port = os.environ.get("SQLSERVER_PORT", "1433")
db_user = os.environ.get("SQLSERVER_USER", "sa")
db_password = os.environ.get("SQLSERVER_PASSWORD", "YourStrongPassword123!")
db_name = os.environ.get("SQLSERVER_DB", "final_project")

jdbc_url = f"jdbc:sqlserver://{host}:{port};databaseName={db_name};encrypt=true;trustServerCertificate=true"
jdbc_driver = "com.microsoft.sqlserver.jdbc.SQLServerDriver"

In [2]:
spark = SparkSession.builder \
    .appName("LoanDefaultPrediction-Pipeline") \
    .config("spark.jars.packages", "com.microsoft.sqlserver:mssql-jdbc:12.6.1.jre11") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [3]:
df = spark.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("driver", jdbc_driver) \
    .option("dbtable", "loans") \
    .option("user", db_user) \
    .option("password", db_password) \
    .load()

## 1. Summary Statistics & Correlation Analysis

In [4]:
numeric_cols = [
    "loan_amnt", "funded_amnt", "term", "int_rate", "installment",
    "annual_inc", "dti", "fico_range_low", "fico_range_high",
    "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc", "mort_acc"
]
df.select(numeric_cols + ["default"]).describe().show()

pairs = [
    ("int_rate", "fico_range_low"),
    ("int_rate", "dti"),
    ("int_rate", "default"),
    ("loan_amnt", "annual_inc"),
    ("dti", "default")
]
for col_a, col_b in pairs:
    corr = df.stat.corr(col_a, col_b)
    print(f"Correlation between {col_a} and {col_b}: {corr:.4f}")

+-------+-----------------+---------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+
|summary|        loan_amnt|    funded_amnt|              term|          int_rate|       installment|        annual_inc|               dti|   fico_range_low|   fico_range_high|         open_acc|           pub_rec|         revol_bal|        revol_util|         total_acc|          mort_acc|           default|
+-------+-----------------+---------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+
|  count|            10000|          10000|             10000|             1

## 2. Groupby Aggregations & Database Export

In [5]:
by_grade = df.groupBy("grade") \
    .agg(
        F.count("*").alias("loan_count"),
        F.round(F.avg("int_rate"), 2).alias("avg_int_rate"),
        F.round(F.avg("loan_amnt"), 2).alias("avg_loan_amnt"),
        F.round(F.avg("default"), 4).alias("default_rate")
    ) \
    .orderBy("grade")

by_grade.show()

# Write aggregation results back to SQL Server for Power BI/Tableau visualization
by_grade.write.format("jdbc") \
    .option("url", jdbc_url) \
    .option("driver", jdbc_driver) \
    .option("dbtable", "processed_results") \
    .option("user", db_user) \
    .option("password", db_password) \
    .mode("overwrite") \
    .save()

+-----+----------+------------+-------------+------------+
|grade|loan_count|avg_int_rate|avg_loan_amnt|default_rate|
+-----+----------+------------+-------------+------------+
|    A|      1755|         7.1|     14180.34|       0.065|
|    B|      2912|       10.68|     13390.93|      0.1405|
|    C|      2776|       13.99|     14349.69|       0.245|
|    D|      1539|        17.7|     15225.42|      0.3216|
|    E|       711|       21.14|     18004.43|       0.384|
|    F|       247|       24.84|     18472.06|      0.4777|
|    G|        60|       27.48|     20983.33|      0.5667|
+-----+----------+------------+-------------+------------+



## 3. Advanced Feature Engineering

In [6]:
# Construct engineered features
processed_df = df.withColumn("fico_average", (F.col("fico_range_low") + F.col("fico_range_high")) / 2.0) \
    .withColumn("loan_to_income", F.col("loan_amnt") / (F.col("annual_inc") + 1.0)) \
    .withColumn("installment_to_income", F.col("installment") / ((F.col("annual_inc") / 12.0) + 1.0)) \
    .withColumn("revol_to_income", F.col("revol_bal") / (F.col("annual_inc") + 1.0)) \
    .withColumn("open_to_total_acc", F.col("open_acc") / (F.col("total_acc") + 1.0)) \
    .withColumn("grade_val", F.ascii(F.substring(F.col("sub_grade"), 1, 1)) - 65) \
    .withColumn("sub_grade_num", F.col("grade_val") * 5 + F.substring(F.col("sub_grade"), 2, 1).cast("int"))

numeric_features = [
    "loan_amnt", "funded_amnt", "term", "int_rate", "installment",
    "annual_inc", "dti", "fico_average", "sub_grade_num",
    "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc", "mort_acc",
    "loan_to_income", "installment_to_income", "revol_to_income", "open_to_total_acc",
    "emp_length"
]
categorical_cols = ["home_ownership", "purpose", "verification_status"]

for col_name in categorical_cols:
    indexer = StringIndexer(inputCol=col_name, outputCol=f"{col_name}_idx", handleInvalid="keep")
    processed_df = indexer.fit(processed_df).transform(processed_df)

ohe_inputs = [f"{c}_idx" for c in categorical_cols]
ohe_outputs = [f"{c}_ohe" for c in categorical_cols]
encoder = OneHotEncoder(inputCols=ohe_inputs, outputCols=ohe_outputs)
processed_df = encoder.fit(processed_df).transform(processed_df)

feature_cols = numeric_features + ohe_outputs
    # Scale numeric features
processed_df = VectorAssembler(inputCols=numeric_features, outputCol="numeric_features").transform(processed_df)
scaler = StandardScaler(inputCol="numeric_features", outputCol="scaled_numeric_features", withStd=True, withMean=True)
processed_df = scaler.fit(processed_df).transform(processed_df)

feature_cols = ["scaled_numeric_features"] + ohe_outputs
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
ml_df = assembler.transform(processed_df)

## 4. Machine Learning Model (Tuned ElasticNet Logistic Regression)

In [7]:
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="default",
    regParam=0.01,
    elasticNetParam=0.5,
    maxIter=100,
    threshold=0.25
)
model = lr.fit(train_df)
predictions = model.transform(test_df)

auc = BinaryClassificationEvaluator(labelCol="default", metricName="areaUnderROC").evaluate(predictions)
accuracy = MulticlassClassificationEvaluator(labelCol="default", metricName="accuracy").evaluate(predictions)

print(f"Optimized Logistic Regression Model Performance:")
print(f"Test AUC: {auc:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Optimized Logistic Regression Model Performance:
Test AUC: 0.7008
Test Accuracy: 0.8053


## 5. Exporting Model Predictions back to SQL Server

In [8]:
# Generate predictions for the entire dataset
full_predictions = model.transform(ml_df)

# Extract prediction label and the default class probability (index 1 of the probability vector)
final_predictions_df = full_predictions.withColumn(
    "predicted_default", F.col("prediction").cast("int")
).withColumn(
    "default_probability", F.round(vector_to_array(F.col("probability"))[1], 4)
).select(df.columns + ["predicted_default", "default_probability"])

print("Writing loans predictions table back to SQL Server...")
final_predictions_df.write.format("jdbc") \
    .option("url", jdbc_url) \
    .option("driver", jdbc_driver) \
    .option("dbtable", "loans_predictions") \
    .option("user", db_user) \
    .option("password", db_password) \
    .mode("overwrite") \
    .save()

print("Exported loans_predictions to SQL Server successfully!")

Writing loans predictions table back to SQL Server...
Exported loans_predictions to SQL Server successfully!


In [9]:
os.makedirs("../outputs", exist_ok=True)
with open("../outputs/model_metrics.txt", "w") as f:
    f.write("Optimized Logistic Regression Classifier\n")
    f.write(f"Test AUC: {auc:.4f}\n")
    f.write(f"Test Accuracy: {accuracy:.4f}\n")

spark.stop()